In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q linear-attention-transformer
!wget -q https://raw.githubusercontent.com/ermongroup/CSDI/main/diff_models.py -O diff_models.py

### Model and Loss

In [ ]:
"""Single-file study pipeline with inlined adapter and original CSDI denoiser.

Colab setup (once):
    !pip -q install linear-attention-transformer
    !wget -q https://raw.githubusercontent.com/ermongroup/CSDI/main/diff_models.py -O diff_models.py

Then run this file. No csdi_official_architecture_wrapper module is needed.
The network diff_CSDI is imported unchanged from the original authors' diff_models.py.
"""

import numpy as np
import torch
from torch import nn
from diff_models import diff_CSDI


class OfficialCSDIImputer(nn.Module):
    def __init__(self, seq_len=1080, d_in=2, channels=64, n_layers=4,
                 n_heads=8, diffusion_steps=50, beta_start=0.0001,
                 beta_end=0.5, schedule='quad', timeemb=128, featureemb=16,
                 diffusion_embedding_dim=128, normalize=True, dt=5.0):
        super().__init__()
        if schedule not in ('quad', 'linear'):
            raise ValueError('schedule must be quad or linear')
        self.seq_len, self.d_in, self.normalize, self.dt = seq_len, d_in, normalize, dt
        self.num_steps = diffusion_steps
        self.timeemb, self.featureemb = timeemb, featureemb
        self.feature_embedding = nn.Embedding(d_in, featureemb)
        cfg = dict(channels=channels, layers=n_layers, nheads=n_heads,
                   diffusion_embedding_dim=diffusion_embedding_dim,
                   num_steps=diffusion_steps,
                   side_dim=timeemb + featureemb + 1, is_linear=False)
        self.diffmodel = diff_CSDI(cfg, inputdim=2)
        if schedule == 'quad':
            betas = np.linspace(np.sqrt(beta_start), np.sqrt(beta_end), diffusion_steps)**2
        else:
            betas = np.linspace(beta_start, beta_end, diffusion_steps)
        betas = torch.tensor(betas, dtype=torch.float32)
        alpha_hat = 1 - betas
        alpha = alpha_hat.cumprod(0)
        self.register_buffer('beta', betas)
        self.register_buffer('alpha_hat', alpha_hat)
        self.register_buffer('alpha', alpha)

    def _check(self, x, mask):
        if x.ndim != 3 or x.shape[1:] != (self.seq_len, self.d_in):
            raise ValueError(f'Expected (B,{self.seq_len},{self.d_in}), got {tuple(x.shape)}')
        if mask.shape != x.shape:
            raise ValueError('mask shape must equal data shape')

    def _normalize(self, x, mask):
        mask = mask.float()
        if not self.normalize:
            mu = torch.zeros(x.size(0), 1, x.size(2), dtype=x.dtype, device=x.device)
            sigma = torch.ones_like(mu)
        else:
            count = mask.sum(dim=1, keepdim=True).clamp_min(1)
            mu = (x * mask).sum(dim=1, keepdim=True) / count
            var = (((x - mu) * mask)**2).sum(dim=1, keepdim=True) / count
            sigma = (var + 1e-5).sqrt()
        return mu, sigma

    def _side_info(self, cond_mask, times=None):
        B, K, L = cond_mask.shape
        if times is None:
            times = torch.arange(L, device=cond_mask.device, dtype=torch.float32)[None].expand(B,-1)
        if times.shape != (B,L):
            raise ValueError('times must have shape (B,T)')
        times = times.to(device=cond_mask.device, dtype=torch.float32)
        frequencies = torch.exp(-torch.log(torch.tensor(10000., device=times.device)) *
                                torch.arange(0,self.timeemb,2,device=times.device)/self.timeemb)
        phases = times.unsqueeze(-1)*frequencies
        time_embedding = torch.zeros(B,L,self.timeemb,device=times.device)
        time_embedding[...,0::2] = phases.sin()
        time_embedding[...,1::2] = phases.cos()
        te = time_embedding.unsqueeze(2).expand(B,L,K,self.timeemb)
        fe = self.feature_embedding(torch.arange(K,device=times.device))
        fe = fe.view(1,1,K,self.featureemb).expand(B,L,K,self.featureemb)
        side = torch.cat((te,fe),dim=-1).permute(0,3,2,1)
        return torch.cat((side, cond_mask.unsqueeze(1)),dim=1)

    @staticmethod
    def _network_input(noisy, clean, cond_mask):
        return torch.stack((cond_mask*clean, (1-cond_mask)*noisy),dim=1)

    def training_loss(self, x_true, m_hat, i_mask, times=None, fixed_step=None):
        """Fixed artificial holdout training. m_hat=visible, i_mask=hidden with GT.

        Note: this differs from the author's original random/historical training mask generator.
        """
        self._check(x_true, m_hat)
        self._check(x_true, i_mask)
        mask = m_hat.float()
        target = i_mask.float()*(1-mask)
        if (mask*target).any():
            raise ValueError('overlapping conditioning and target positions')
        if target.sum() == 0:
            raise ValueError('no supervised target values')
        mu,sigma = self._normalize(x_true*mask,mask)
        normalized = (x_true-mu)/sigma
        normalized = torch.where((mask+target)>0, normalized, 0.)
        clean = normalized.permute(0,2,1)
        cm = mask.permute(0,2,1)
        tm = target.permute(0,2,1)
        B = x_true.size(0)
        if fixed_step is None:
            step = torch.randint(self.num_steps,(B,),device=x_true.device)
        else:
            step = torch.full((B,),int(fixed_step),device=x_true.device,dtype=torch.long)
        noise = torch.randn_like(clean)
        abar = self.alpha[step].view(B,1,1)
        noisy = abar.sqrt()*clean + (1-abar).sqrt()*noise
        side = self._side_info(cm,times)
        predicted = self.diffmodel(self._network_input(noisy,clean,cm),side,step)
        return (((predicted-noise)*tm)**2).sum()/tm.sum().clamp_min(1.)

    @torch.no_grad()
    def sample(self, x_in, m_hat, n_samples=1, times=None):
        """Return (n_samples,B,T,D) in original units, sampled independently."""
        self._check(x_in,m_hat)
        if n_samples < 1: raise ValueError('n_samples >= 1 required')
        mask = m_hat.float()
        mu,sigma = self._normalize(x_in*mask,mask)
        clean = (((x_in-mu)/sigma)*mask).permute(0,2,1)
        cm = mask.permute(0,2,1)
        side = self._side_info(cm,times)
        B,K,L = clean.shape
        results=[]
        for _ in range(n_samples):
            current = torch.randn_like(clean)
            for t in range(self.num_steps-1,-1,-1):
                step = torch.full((B,),t,device=x_in.device,dtype=torch.long)
                eps_hat = self.diffmodel(self._network_input(current,clean,cm),side,step)
                current = (current-(1-self.alpha_hat[t])/torch.sqrt(1-self.alpha[t])*eps_hat) / torch.sqrt(self.alpha_hat[t])
                if t>0:
                    sigma_t = torch.sqrt((1-self.alpha[t-1])/(1-self.alpha[t])*self.beta[t])
                    current = current + sigma_t*torch.randn_like(current)
            sample = current.permute(0,2,1)*sigma+mu
            results.append(mask*x_in+(1-mask)*sample)
        return torch.stack(results,dim=0)

    @torch.no_grad()
    def forward(self, x_in, m_hat, n_samples=1, times=None):
        """MTSIT-compatible 5 outputs; n_samples>1 uses pointwise median."""
        samples = self.sample(x_in,m_hat,n_samples=n_samples,times=times)
        x_c = samples.median(dim=0).values if n_samples>1 else samples[0]
        mask = m_hat.float()
        mu,sigma = self._normalize(x_in*mask,mask)
        x_hat = x_c  # at observed positions x_c retains observed values
        x_norm_hat = (x_hat-mu)/sigma
        return x_hat,x_c,x_norm_hat,mu,sigma

### Model training, validation, and testing

In [ ]:
NPZ_PATH = '/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
OUT_DIR = '/content/drive/MyDrive/aim3/CSDI_1e-3_16'

In [ ]:
"""Official CSDI denoiser: patient-level five-fold CV.
Requires only the authors' official diff_models.py in the Colab working directory.
The core diff_CSDI network is from the authors; masking and per-segment normalization are study-specific adaptations.
Expected NPZ keys:
 X_in   (N,T,D) masked input, usually zero-filled
 M_hat  (N,T,D) 1 where visible to model
 X      (N,T,D) ground truth (NaN allowed outside valid observations)
 I      (N,T,D) 1 for artificially held-out EVALUATION targets
 segment_ids (N,) patient IDs
The artificial holdout I must be disjoint from M_hat.
"""
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ---------------------------- Configuration ----------------------------

SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
EPOCHS = 300
EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE = 16  # 1080-step temporal attention is memory-intensive
LR = 1e-3  # safer starting learning rate for diffusion
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEQ_LEN = 1080
D_IN = 2
CHANNELS = 64
N_HEADS = 8
N_LAYERS = 4
N_SAMPLES = 10 
DIFFUSION_STEPS = 50
DT_SECONDS = 5.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PIN_MEMORY = DEVICE == 'cuda'
TEST_SAMPLE_SEED = SEED + 7000

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class TSImputeDataset(Dataset):
    def __init__(self, x_in, obs, ground_truth, holdout, indices):
        self.x_in = torch.from_numpy(np.nan_to_num(x_in, nan=0, posinf=0, neginf=0).astype('float32'))
        self.obs = torch.from_numpy(obs.astype('float32'))
        self.truth = torch.from_numpy(np.nan_to_num(ground_truth, nan=0, posinf=0, neginf=0).astype('float32'))
        self.target = torch.from_numpy(holdout.astype('float32'))
        self.indices = torch.from_numpy(indices.astype('int64'))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.x_in[i], self.obs[i], self.truth[i], self.target[i], self.indices[i]


def build_loader(idx, shuffle=False):
    ds = TSImputeDataset(x_in[idx], m_hat[idx], x_true[idx], i_mask[idx], idx)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=PIN_MEMORY)

def diffusion_loss_given_holdout(model, x_true_b, obs_b, target_b):
    """CSDI conditional noise-prediction MSE on fixed artificial targets."""
    eligible = target_b * (1.0 - obs_b)
    if eligible.sum().item() == 0:
        return None
    return model.training_loss(x_true_b, obs_b, eligible)


def run_loss_epoch(model, loader, optimizer=None, eval_seed=None):
    training = optimizer is not None
    model.train(training)
    losses, total_weights = 0., 0.
    # Make validation noise/steps identical across epochs; testing samples have separate seed.
    if eval_seed is not None:
        cpu_state = torch.random.get_rng_state()
        cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        seed_all(eval_seed)
    try:
        with torch.set_grad_enabled(training):
            for _, obs, truth, target, _ in loader:
                obs = obs.to(DEVICE, non_blocking=True)
                truth = truth.to(DEVICE, non_blocking=True)
                target = target.to(DEVICE, non_blocking=True)
                if optimizer is not None:
                    optimizer.zero_grad(set_to_none=True)
                loss = diffusion_loss_given_holdout(model, truth, obs, target)
                if loss is None:
                    continue
                if training:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
                    optimizer.step()
                weight = target.sum().item()
                losses += float(loss.detach()) * weight
                total_weights += weight
    finally:
        if eval_seed is not None:
            torch.random.set_rng_state(cpu_state)
            if cuda_states is not None:
                torch.cuda.set_rng_state_all(cuda_states)
    return losses / total_weights if total_weights else float('nan')


def gap_lengths(mask):
    m = np.asarray(mask) > 0.5
    changes = np.flatnonzero(np.diff(np.r_[False, m, False]))
    lengths = np.zeros(len(m), dtype=np.int32)
    for start, end in zip(changes[::2], changes[1::2]):
        lengths[start:end] = end - start
    return lengths


def gap_region(g):
    # dt=5 seconds; left inclusive/right exclusive in actual minutes.
    if 1 <= g < 12: return '<1 min'
    if 12 <= g < 120: return '[1,10) min'
    if 120 <= g < 240: return '[10,20) min'
    if 240 <= g < 360: return '[20,30] min'
    if g > 360: return '>30 min'
    return None


def collect_test_predictions(model, loader, fold):
    model.eval()
    rows = []
    with torch.no_grad():
        for x_b, m_b, truth_b, target_b, idx_b in loader:
            x_b = x_b.to(DEVICE)
            m_b = m_b.to(DEVICE)
            _, completed, _, _, _ = model(x_b, m_b, n_samples=N_SAMPLES)
            preds = completed.cpu().numpy()
            targets = target_b.numpy()
            truth = truth_b.numpy()
            for b, idx_tensor in enumerate(idx_b):
                idx = int(idx_tensor)
                for d in range(D_IN):
                    lengths = gap_lengths(targets[b, :, d])
                    for t in np.flatnonzero(targets[b, :, d] > 0.5):
                        g = int(lengths[t])
                        y = float(truth[b, t, d])
                        p = float(preds[b, t, d])
                        rows.append({'fold': fold, 'model': 'CSDI_official_architecture',
                                     'segment_idx': idx, 'patient_id': str(seg_id[idx]),
                                     'filename': str(filenames[idx]), 'time_idx': int(t),
                                     'target': str(target_names[d]), 'y_true': y,
                                     'y_pred': p, 'abs_err': abs(y-p),
                                     'gap_len': g, 'gap_region': gap_region(g)})
    return pd.DataFrame(rows)


def add_metrics(df, keys):
    if df.empty:
        return pd.DataFrame()
    output = []
    for names, group in df.groupby(keys, dropna=False):
        if not isinstance(names, tuple):
            names = (names,)
        error = group.y_pred.to_numpy() - group.y_true.to_numpy()
        output.append(dict(zip(keys, names), test_mae=float(np.mean(abs(error))),
                           test_rmse=float(np.sqrt(np.mean(error**2))), n_points=len(group)))
    return pd.DataFrame(output)


def summarize_folds(metrics, group_keys):
    if metrics.empty:
        return pd.DataFrame()
    return metrics.groupby(group_keys, as_index=False).agg(
        test_mae_mean=('test_mae','mean'), test_mae_std=('test_mae','std'),
        test_rmse_mean=('test_rmse','mean'), test_rmse_std=('test_rmse','std'),
        n_folds=('fold','nunique'), total_points=('n_points','sum'))


def main():
    global x_in, m_hat, x_true, i_mask, seg_id, filenames, target_names
    seed_all(SEED)
    out = Path(OUT_DIR)
    (out / 'ckpts').mkdir(parents=True, exist_ok=True)
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        required = ['X_in','M_hat','X','I','segment_ids']
        missing = [key for key in required if key not in data.files]
        if missing:
            raise KeyError(f'NPZ missing fields: {missing}')
        x_in = np.asarray(data['X_in'])
        m_hat = np.asarray(data['M_hat'])
        x_true = np.asarray(data['X'])
        i_mask = np.asarray(data['I'])
        seg_id = np.asarray(data['segment_ids'])
        filenames = np.asarray(data['filenames']).astype(str) if 'filenames' in data.files else np.array([f'segment_{i}' for i in range(len(seg_id))])
        key = 'target_columns' if 'target_columns' in data.files and len(data['target_columns']) == D_IN else 'columns_all'
        target_names = np.asarray(data[key]).astype(str).tolist() if key in data.files and len(data[key]) == D_IN else [f'channel_{d}' for d in range(D_IN)]
    if not (x_in.shape == m_hat.shape == x_true.shape == i_mask.shape):
        raise ValueError('X_in/M_hat/X/I must share identical shapes')
    N, T, D = x_in.shape
    if (T,D) != (SEQ_LEN,D_IN):
        raise ValueError(f'Expected (*,{SEQ_LEN},{D_IN}), found {x_in.shape}')
    if len(seg_id) != N:
        raise ValueError('segment_ids must be one per segment')
    if not np.isin(m_hat, [0,1]).all() or not np.isin(i_mask, [0,1]).all():
        raise ValueError('M_hat and I must be binary masks')
    if np.any((m_hat > 0) & (i_mask > 0)):
        raise ValueError('Artificial target I must be disjoint from observed mask M_hat')
    if not np.isfinite(x_true[i_mask > 0]).all():
        raise ValueError('Artificial target locations in X must have finite ground truth')
    if not np.isfinite(x_true[m_hat > 0]).all():
        raise ValueError('Observed locations in X must have finite ground truth')
    if not np.isfinite(x_in[m_hat > 0]).all():
        raise ValueError('Observed locations in X_in must have finite input')
    # Ensure prefilled input agrees with original observations.
    if not np.allclose(x_in[m_hat > 0], x_true[m_hat > 0], rtol=1e-4, atol=1e-4):
        raise ValueError('X_in and X differ at observed locations. Verify NPZ input preprocessing.')
    print('Device:', DEVICE, '| Dataset:', x_in.shape, '| Patients:',len(np.unique(seg_id)))
    pats = np.unique(seg_id)
    if len(pats) < N_SPLITS + 1:
        raise ValueError('Not enough distinct patients for train/val/test splitting')
    rng = np.random.default_rng(SEED)
    rng.shuffle(pats)
    folds = np.array_split(pats, N_SPLITS)
    pred_tables, fold_splits, fold_details = [], [], []
    for fold, test_patients in enumerate(folds, 1):
        other = pats[~np.isin(pats, test_patients)].copy()
        vrng = np.random.default_rng(SEED + fold)
        vrng.shuffle(other)
        n_val = max(1, min(len(other)-1, round(len(other)*VAL_WITHIN_TRAIN_P)))
        val_patients, train_patients = other[:n_val], other[n_val:]
        train_idx = np.flatnonzero(np.isin(seg_id, train_patients))
        val_idx = np.flatnonzero(np.isin(seg_id, val_patients))
        test_idx = np.flatnonzero(np.isin(seg_id, test_patients))
        fold_splits.append(dict(fold=fold,
            train_patients=','.join(map(str,train_patients)),val_patients=','.join(map(str,val_patients)),
            test_patients=','.join(map(str,test_patients)),
            n_train_patients=len(train_patients),n_val_patients=len(val_patients),n_test_patients=len(test_patients),
            n_train_segments=len(train_idx),n_val_segments=len(val_idx),n_test_segments=len(test_idx)))
        print(f'\nFOLD {fold}/{N_SPLITS}: train/val/test segments = {len(train_idx)}/{len(val_idx)}/{len(test_idx)}', flush=True)
        tr, va, te = build_loader(train_idx, True), build_loader(val_idx), build_loader(test_idx)
        seed_all(SEED+fold)
        model = OfficialCSDIImputer(seq_len=T, d_in=D, channels=CHANNELS,
                            n_layers=N_LAYERS, n_heads=N_HEADS, diffusion_steps=DIFFUSION_STEPS,
                            normalize=True, dt=DT_SECONDS).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
        ckpt = out / 'ckpts' / f'fold{fold:02d}_best.pt'
        best, best_ep, no_improve = float('inf'), 0, 0
        for ep in range(1, EPOCHS+1):
            train_loss = run_loss_epoch(model, tr, opt)
            val_loss = run_loss_epoch(model, va, eval_seed=SEED+10000+fold)
            if not np.isfinite(val_loss):
                raise ValueError('Validation fold has zero artificial holdout points or nonfinite loss')
            scheduler.step(val_loss)
            print(f'Epoch {ep:03d} | train denoise MSE={train_loss:.5f} | val denoise MSE={val_loss:.5f}', flush=True)
            if val_loss < best:
                best, best_ep, no_improve = val_loss, ep, 0
                torch.save(model.state_dict(),ckpt)
            else:
                no_improve += 1
                if no_improve >= EARLY_STOPPING_PATIENCE:
                    break
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        test_denoise_loss = run_loss_epoch(model, te, eval_seed=SEED+20000+fold)
        # Test MAE/RMSE requires actual diffusion sampling, not denoising MSE.
        seed_all(TEST_SAMPLE_SEED+fold)
        fold_pred = collect_test_predictions(model, te, fold)
        if fold_pred.empty:
            raise ValueError(f'Fold {fold} has no artificial missing target points')
        pred_tables.append(fold_pred)
        fold_details.append(dict(fold=fold,best_epoch=best_ep,val_denoising_mse=best,
                                 test_denoising_mse=test_denoise_loss))
        fold_pred.to_csv(out/f'fold{fold:02d}_predictions.csv',index=False)
        print(f'Fold {fold} best epoch {best_ep}; val denoising MSE {best:.5f}; collected {len(fold_pred)} masked points', flush=True)
        for channel, frame in fold_pred.groupby('target'):
            err = frame.y_pred.to_numpy() - frame.y_true.to_numpy()
            eligible = np.abs(frame.y_true.to_numpy()) > 1e-8
            mape = 100*np.mean(np.abs(err[eligible]/frame.y_true.to_numpy()[eligible])) if eligible.any() else float('nan')
            print(f'  {channel}: TEST MAE={np.mean(np.abs(err)):.4f}, RMSE={np.sqrt(np.mean(err**2)):.4f}, MAPE={mape:.3f}%', flush=True)
    predictions = pd.concat(pred_tables, ignore_index=True)
    predictions.to_csv(out/'cv5_test_predictions_pooled.csv',index=False)
    pd.DataFrame(fold_splits).to_csv(out/'cv5_patient_split_summary.csv',index=False)
    fold_metrics = add_metrics(predictions, ['fold','target','model'])
    fold_metrics = fold_metrics.merge(pd.DataFrame(fold_details), on='fold', how='left')
    fold_metrics.to_csv(out/'cv5_fold_results.csv',index=False)
    summarize_folds(fold_metrics,['target','model']).to_csv(out/'cv5_summary_mean_std.csv',index=False)
    pooled = add_metrics(predictions,['target','model']).rename(columns={'test_mae':'pooled_test_mae','test_rmse':'pooled_test_rmse'})
    pooled.to_csv(out/'cv5_pooled_test_metrics.csv',index=False)
    gap_preds = predictions[predictions.gap_region.notna()].copy()
    gap_preds.to_csv(out/'cv5_test_predictions_pooled_with_gap_region.csv',index=False)
    pooled_gap = add_metrics(gap_preds,['target','model','gap_region']).rename(columns={'test_mae':'pooled_test_mae','test_rmse':'pooled_test_rmse'})
    pooled_gap.to_csv(out/'cv5_pooled_test_metrics_by_gap_region.csv',index=False)
    fold_gap = add_metrics(gap_preds,['fold','target','model','gap_region'])
    fold_gap.to_csv(out/'cv5_fold_metrics_by_gap_region.csv',index=False)
    summarize_folds(fold_gap,['target','model','gap_region']).to_csv(out/'cv5_summary_mean_std_by_gap_region.csv',index=False)
    print('\nFold metrics:\n',fold_metrics.to_string(index=False))
    print('\nPooled metrics:\n',pooled.to_string(index=False))
    print('\nGap metrics:\n',pooled_gap.to_string(index=False))
    print('\nSaved all outputs to:',out)

main()

Device: cuda | Dataset: (3817, 1080, 2) | Patients: 64

FOLD 1/5: train/val/test segments = 2657/413/747
Epoch 001 | train denoise MSE=0.51090 | val denoise MSE=0.44672
Epoch 002 | train denoise MSE=0.42272 | val denoise MSE=0.44445
Epoch 003 | train denoise MSE=0.39675 | val denoise MSE=0.44247
Epoch 004 | train denoise MSE=0.39307 | val denoise MSE=0.41205
Epoch 005 | train denoise MSE=0.40187 | val denoise MSE=0.40802
Epoch 006 | train denoise MSE=0.37886 | val denoise MSE=0.40654
Epoch 007 | train denoise MSE=0.37409 | val denoise MSE=0.41184
Epoch 008 | train denoise MSE=0.38290 | val denoise MSE=0.40263
Epoch 009 | train denoise MSE=0.37169 | val denoise MSE=0.42298
Epoch 010 | train denoise MSE=0.34855 | val denoise MSE=0.39062
Epoch 011 | train denoise MSE=0.38224 | val denoise MSE=0.38710
Epoch 012 | train denoise MSE=0.36465 | val denoise MSE=0.39027
Epoch 013 | train denoise MSE=0.36005 | val denoise MSE=0.38073
Epoch 014 | train denoise MSE=0.36704 | val denoise MSE=0.37852

/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Epoch 001 | train denoise MSE=0.50883 | val denoise MSE=0.38753
Epoch 002 | train denoise MSE=0.39947 | val denoise MSE=0.36778
Epoch 003 | train denoise MSE=0.37960 | val denoise MSE=0.36889
Epoch 004 | train denoise MSE=0.40150 | val denoise MSE=0.36536
Epoch 005 | train denoise MSE=0.36575 | val denoise MSE=0.35678
Epoch 006 | train denoise MSE=0.37614 | val denoise MSE=0.34513
Epoch 007 | train denoise MSE=0.40395 | val denoise MSE=0.36307
Epoch 008 | train denoise MSE=0.33831 | val denoise MSE=0.34942
Epoch 009 | train denoise MSE=0.35864 | val denoise MSE=0.33861
Epoch 010 | train denoise MSE=0.35546 | val denoise MSE=0.32927
Epoch 011 | train denoise MSE=0.37688 | val denoise MSE=0.33266
Epoch 012 | train denoise MSE=0.38021 | val denoise MSE=0.33814
Epoch 013 | train denoise MSE=0.35331 | val denoise MSE=0.32779
Epoch 014 | train denoise MSE=0.37368 | val denoise MSE=0.32157
Epoch 015 | train denoise MSE=0.34898 | val denoise MSE=0.33546
Epoch 016 | train denoise MSE=0.36406 | 

/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Epoch 001 | train denoise MSE=0.52191 | val denoise MSE=0.55292
Epoch 002 | train denoise MSE=0.41521 | val denoise MSE=0.51313
Epoch 003 | train denoise MSE=0.39645 | val denoise MSE=0.49545
Epoch 004 | train denoise MSE=0.41259 | val denoise MSE=0.50428
Epoch 005 | train denoise MSE=0.38814 | val denoise MSE=0.49822
Epoch 006 | train denoise MSE=0.39159 | val denoise MSE=0.48346
Epoch 007 | train denoise MSE=0.36683 | val denoise MSE=0.49272
Epoch 008 | train denoise MSE=0.38778 | val denoise MSE=0.48635
Epoch 009 | train denoise MSE=0.37719 | val denoise MSE=0.48047
Epoch 010 | train denoise MSE=0.38041 | val denoise MSE=0.47203
Epoch 011 | train denoise MSE=0.36837 | val denoise MSE=0.48120
Epoch 012 | train denoise MSE=0.35965 | val denoise MSE=0.47867
Epoch 013 | train denoise MSE=0.38571 | val denoise MSE=0.47100
Epoch 014 | train denoise MSE=0.34143 | val denoise MSE=0.48690
Epoch 015 | train denoise MSE=0.34896 | val denoise MSE=0.46271
Epoch 016 | train denoise MSE=0.35741 | 

/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Epoch 001 | train denoise MSE=0.53067 | val denoise MSE=0.43905
Epoch 002 | train denoise MSE=0.44052 | val denoise MSE=0.38918
Epoch 003 | train denoise MSE=0.38623 | val denoise MSE=0.41307
Epoch 004 | train denoise MSE=0.40421 | val denoise MSE=0.38508
Epoch 005 | train denoise MSE=0.39769 | val denoise MSE=0.38186
Epoch 006 | train denoise MSE=0.37863 | val denoise MSE=0.37541
Epoch 007 | train denoise MSE=0.37100 | val denoise MSE=0.37970
Epoch 008 | train denoise MSE=0.39033 | val denoise MSE=0.37866
Epoch 009 | train denoise MSE=0.37169 | val denoise MSE=0.37940
Epoch 010 | train denoise MSE=0.36807 | val denoise MSE=0.37472
Epoch 011 | train denoise MSE=0.38681 | val denoise MSE=0.36891
Epoch 012 | train denoise MSE=0.34697 | val denoise MSE=0.37845
Epoch 013 | train denoise MSE=0.34200 | val denoise MSE=0.36400
Epoch 014 | train denoise MSE=0.36237 | val denoise MSE=0.36784
Epoch 015 | train denoise MSE=0.36339 | val denoise MSE=0.36633
Epoch 016 | train denoise MSE=0.35438 | 

/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Epoch 001 | train denoise MSE=0.51609 | val denoise MSE=0.39717
Epoch 002 | train denoise MSE=0.42404 | val denoise MSE=0.38291
Epoch 003 | train denoise MSE=0.41522 | val denoise MSE=0.37115
Epoch 004 | train denoise MSE=0.37362 | val denoise MSE=0.37851
Epoch 005 | train denoise MSE=0.38327 | val denoise MSE=0.35943
Epoch 006 | train denoise MSE=0.39955 | val denoise MSE=0.37145
Epoch 007 | train denoise MSE=0.40209 | val denoise MSE=0.36120
Epoch 008 | train denoise MSE=0.33220 | val denoise MSE=0.35761
Epoch 009 | train denoise MSE=0.37596 | val denoise MSE=0.36637
Epoch 010 | train denoise MSE=0.36383 | val denoise MSE=0.36005
Epoch 011 | train denoise MSE=0.37872 | val denoise MSE=0.35054
Epoch 012 | train denoise MSE=0.37976 | val denoise MSE=0.35118
Epoch 013 | train denoise MSE=0.37813 | val denoise MSE=0.34787
Epoch 014 | train denoise MSE=0.37656 | val denoise MSE=0.35226
Epoch 015 | train denoise MSE=0.39313 | val denoise MSE=0.34726
Epoch 016 | train denoise MSE=0.37017 | 

### Validation performance

In [ ]:

import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

NPZ_PATH = '/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
OUT_DIR = '/content/drive/MyDrive/aim3/CSDI_1e-3_16'
SPLIT_CSV = os.path.join(OUT_DIR, 'cv5_patient_split_summary.csv')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
SEQ_LEN = 1080
D_IN = 2
CHANNELS = 64
N_LAYERS = 4
N_HEADS = 8
DIFFUSION_STEPS = 50
BATCH_SIZE = 16 
N_SAMPLES = 10 
VAL_LOSS_REPEATS = 3 
SEED = 1337
EPS = 1e-8
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
os.makedirs(OUT_DIR, exist_ok=True)

class ValidationDataset(Dataset):
    def __init__(self, x_in, observed, truth, target, global_indices):
        self.x_in = torch.from_numpy(np.nan_to_num(x_in, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32))
        self.observed = torch.from_numpy(observed.astype(np.float32))
        self.truth = torch.from_numpy(np.nan_to_num(truth, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32))
        self.target = torch.from_numpy(target.astype(np.float32))
        self.global_indices = np.asarray(global_indices, dtype=np.int64)
    def __len__(self): return len(self.global_indices)
    def __getitem__(self, i):
        return self.x_in[i], self.observed[i], self.truth[i], self.target[i], int(self.global_indices[i])

def parse_patients(v):
    if pd.isna(v) or str(v).strip() == '': return []
    return [int(float(s.strip())) for s in str(v).split(',') if s.strip()]

def gap_lengths(mask):
    a = np.asarray(mask) > .5
    padded = np.r_[False, a, False]
    start = np.flatnonzero(np.diff(padded.astype('int8')) == 1)
    end = np.flatnonzero(np.diff(padded.astype('int8')) == -1)
    lengths = np.zeros(a.size, dtype=np.int32)
    for s,e in zip(start,end): lengths[s:e] = e-s
    return lengths

def gap_region(g):
    if 1 <= g <= 11: return '<1 min'
    if 12 <= g <= 119: return '[1,10) min'
    if 120 <= g <= 239: return '[10,20) min'
    if 240 <= g <= 359: return '[20,30) min'
    if g == 360: return '=30 min'
    return None

def metrics(df):
    if len(df) == 0: return {'mae':np.nan,'rmse':np.nan,'mape_pct':np.nan,'n_points':0,'n_mape':0}
    yt=df['y_true'].to_numpy(dtype=float)
    yp=df['y_pred'].to_numpy(dtype=float)
    delta=yp-yt
    valid=np.abs(yt)>EPS
    return dict(mae=float(np.abs(delta).mean()),rmse=float(np.sqrt(np.square(delta).mean())),
                mape_pct=float(100*np.mean(np.abs(delta[valid]/yt[valid]))) if valid.any() else np.nan,
                n_points=len(df),n_mape=int(valid.sum()))

with np.load(NPZ_PATH, allow_pickle=True) as data:
    x_in = data['X_in']; observed = data['M_hat']; truth = data['X']; target = data['I']
    seg_id = data['segment_ids']
    filenames = np.array([str(x) for x in data['filenames']]) if 'filenames' in data else np.array([f'segment_{i}' for i in range(len(seg_id))])
    if 'target_columns' in data and len(data['target_columns']) == x_in.shape[-1]:
        target_names = [str(x) for x in data['target_columns']]
    else:
        target_names = [f'channel_{i}' for i in range(x_in.shape[-1])]
assert x_in.shape == observed.shape == truth.shape == target.shape
assert x_in.shape[1:] == (SEQ_LEN, D_IN), x_in.shape
assert np.all((observed > .5) & (target > .5) == False), 'Target and observed masks overlap'
splits = pd.read_csv(SPLIT_CSV)
all_rows=[]; all_loss=[]; all_fold_metrics=[]; all_gap_metrics=[]
for _, r in splits.iterrows():
    fold=int(r['fold'])
    vals=set(parse_patients(r['val_patients']))
    indices=np.flatnonzero(np.isin(seg_id, list(vals)))
    if len(indices)==0: raise ValueError(f'Fold {fold}: no validation segments found')
    loader=DataLoader(ValidationDataset(x_in[indices],observed[indices],truth[indices],target[indices],indices),
                      batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=(DEVICE=='cuda'))
    model=OfficialCSDIImputer(seq_len=SEQ_LEN,d_in=D_IN,channels=CHANNELS,n_layers=N_LAYERS,
                              n_heads=N_HEADS,diffusion_steps=DIFFUSION_STEPS,
                              normalize=True,dt=5.0).to(DEVICE)
    ckpt=os.path.join(CKPT_DIR,f'fold{fold:02d}_best.pt')
    if not os.path.exists(ckpt): raise FileNotFoundError(ckpt)
    model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True))
    model.eval()
    torch.manual_seed(SEED+fold*1000)
    if DEVICE=='cuda': torch.cuda.manual_seed_all(SEED+fold*1000)
    loss_weighted=0.0; target_points=0
    with torch.no_grad():
        for x_b,m_b,y_b,i_b,_ in loader:
            x_b=x_b.to(DEVICE);m_b=m_b.to(DEVICE);y_b=y_b.to(DEVICE);i_b=i_b.to(DEVICE)
            n=float((i_b*(1-m_b)).sum().item())
            if n==0: continue
            for _rep in range(VAL_LOSS_REPEATS):
                loss=model.training_loss(y_b,m_b,i_b)
                loss_weighted += loss.item()*n
            target_points += int(n)
    val_loss=loss_weighted / max(1,target_points*VAL_LOSS_REPEATS)
    all_loss.append(dict(fold=fold,model='CSDI_OfficialArchitecture',val_denoise_mse=val_loss,
                         n_val_patients=len(vals),n_val_segments=len(indices),n_target_points=target_points,
                         repeats=VAL_LOSS_REPEATS))
    torch.manual_seed(SEED+fold*1000+1)
    if DEVICE=='cuda': torch.cuda.manual_seed_all(SEED+fold*1000+1)
    rows=[]
    with torch.no_grad():
        for x_b,m_b,y_b,i_b,global_idx_b in loader:
            x_b=x_b.to(DEVICE);m_b=m_b.to(DEVICE)
            _,completed,_,_,_=model(x_b,m_b,n_samples=N_SAMPLES)
            completed=completed.cpu().numpy();y_b=y_b.numpy();i_b=i_b.numpy()
            for b in range(len(completed)):
                idx=int(global_idx_b[b])
                for d in range(D_IN):
                    gaps=gap_lengths(i_b[b,:,d])
                    for t in np.flatnonzero(i_b[b,:,d]>.5):
                        if not np.isfinite(truth[idx,t,d]): continue
                        g=int(gaps[t]);yt=float(y_b[b,t,d]);yp=float(completed[b,t,d])
                        rows.append(dict(fold=fold,split='val',segment_idx=idx,
                                         patient_id=int(seg_id[idx]),filename=str(filenames[idx]),
                                         time_idx=int(t),target=target_names[d],y_true=yt,y_pred=yp,
                                         abs_err=abs(yp-yt),gap_len=g,gap_region=gap_region(g),
                                         model='CSDI_OfficialArchitecture'))
    df=pd.DataFrame(rows)
    if df.empty: raise ValueError(f'Fold {fold} has no valid validation target points')
    df.to_csv(os.path.join(OUT_DIR,f'fold{fold:02d}_validation_predictions.csv'),index=False)
    all_rows.append(df)
    for target_name, sub in df.groupby('target'):
        all_fold_metrics.append(dict(fold=fold,target=target_name,**metrics(sub)))
    for (target_name,category),sub in df.dropna(subset=['gap_region']).groupby(['target','gap_region']):
        all_gap_metrics.append(dict(fold=fold,target=target_name,gap_region=category,**metrics(sub)))
    print(f'Fold {fold}: validation denoise MSE={val_loss:.6f}, points={target_points}')
    for target_name,sub in df.groupby('target'):
        m=metrics(sub)
        print(f'  {target_name}: MAE={m["mae"]:.5f}, RMSE={m["rmse"]:.5f}, MAPE={m["mape_pct"]:.3f}%')
    del model
    if DEVICE=='cuda': torch.cuda.empty_cache()

val_loss_df=pd.DataFrame(all_loss)
val_loss_df.to_csv(os.path.join(OUT_DIR,'cv5_validation_losses_recomputed.csv'),index=False)
loss_summary=pd.DataFrame([{'model':'CSDI_OfficialArchitecture',
    'val_denoise_mse_mean':val_loss_df.val_denoise_mse.mean(),
    'val_denoise_mse_std':val_loss_df.val_denoise_mse.std(ddof=1),
    'n_folds':len(val_loss_df)}])
loss_summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_loss_summary_mean_std.csv'),index=False)
val_pred_df=pd.concat(all_rows,ignore_index=True)
val_pred_df.to_csv(os.path.join(OUT_DIR,'cv5_validation_predictions_pooled.csv'),index=False)
val_pred_df.dropna(subset=['gap_region']).to_csv(os.path.join(OUT_DIR,'cv5_validation_predictions_pooled_with_gap_region.csv'),index=False)
fold_metrics=pd.DataFrame(all_fold_metrics)
fold_metrics.to_csv(os.path.join(OUT_DIR,'cv5_validation_fold_metrics.csv'),index=False)
gap_metrics=pd.DataFrame(all_gap_metrics)
gap_metrics.to_csv(os.path.join(OUT_DIR,'cv5_validation_fold_metrics_by_gap_region.csv'),index=False)
summary=fold_metrics.groupby('target',as_index=False).agg(
    mae_mean=('mae','mean'),mae_std=('mae','std'),rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
    mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),n_folds=('fold','count'))
summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_metrics_mean_std.csv'),index=False)
pooled=pd.DataFrame([{'target':k,**metrics(s)} for k,s in val_pred_df.groupby('target')])
pooled.to_csv(os.path.join(OUT_DIR,'cv5_validation_pooled_metrics.csv'),index=False)
pooled_gap=pd.DataFrame([{'target':k[0],'gap_region':k[1],**metrics(s)}
    for k,s in val_pred_df.dropna(subset=['gap_region']).groupby(['target','gap_region'])])
pooled_gap.to_csv(os.path.join(OUT_DIR,'cv5_validation_pooled_metrics_by_gap_region.csv'),index=False)
gap_summary=gap_metrics.groupby(['target','gap_region'],as_index=False).agg(
    mae_mean=('mae','mean'),mae_std=('mae','std'),rmse_mean=('rmse','mean'),rmse_std=('rmse','std'),
    mape_mean_pct=('mape_pct','mean'),mape_std_pct=('mape_pct','std'),n_folds=('fold','count'),
    total_points=('n_points','sum'))
gap_summary.to_csv(os.path.join(OUT_DIR,'cv5_validation_metrics_mean_std_by_gap_region.csv'),index=False)
print('\nFive-fold validation denoising MSE:')
print(val_loss_df.to_string(index=False))
print('\nDenoising MSE mean ± SD: {:.6f} ± {:.6f}'.format(loss_summary.val_denoise_mse_mean.iloc[0],loss_summary.val_denoise_mse_std.iloc[0]))
print('\nValidation imputation metrics mean ± SD:')
print(summary.to_string(index=False))
print('\nSaved validation files under:',OUT_DIR)

Device: cuda


/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Fold 1: validation denoise MSE=0.215395, points=14265
  channel_0: MAE=1.22450, RMSE=2.42154, MAPE=2.009%
  channel_1: MAE=1.65879, RMSE=3.63280, MAPE=2.421%


/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Fold 2: validation denoise MSE=0.161560, points=11136
  channel_0: MAE=1.05914, RMSE=1.74589, MAPE=1.851%
  channel_1: MAE=0.72714, RMSE=1.26750, MAPE=1.333%


...

Fold 3: validation denoise MSE=0.171095, points=16210
  channel_0: MAE=1.09917, RMSE=1.82736, MAPE=1.783%
  channel_1: MAE=1.80942, RMSE=3.71323, MAPE=2.691%


/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Fold 4: validation denoise MSE=0.255336, points=8842
  channel_0: MAE=2.17432, RMSE=4.33273, MAPE=4.391%
  channel_1: MAE=2.44951, RMSE=9.62217, MAPE=3.920%


/content/diff_models.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  return nn.TransformerEncoder(encoder_layer, num_layers=layers)


Fold 5: validation denoise MSE=0.206836, points=14246
  channel_0: MAE=1.36505, RMSE=2.02318, MAPE=2.440%
  channel_1: MAE=0.87469, RMSE=3.16785, MAPE=1.611%

Five-fold validation denoising MSE:
 fold                     model  val_denoise_mse  n_val_patients  n_val_segments  n_target_points  repeats
    1 CSDI_OfficialArchitecture         0.215395               5             413            14265        3
    2 CSDI_OfficialArchitecture         0.161560               5             293            11136        3
    3 CSDI_OfficialArchitecture         0.171095               5             402            16210        3
    4 CSDI_OfficialArchitecture         0.255336               5             281             8842        3
    5 CSDI_OfficialArchitecture         0.206836               5             345            14246        3

Denoising MSE mean ± SD: 0.202044 ± 0.037543

Validation imputation metrics mean ± SD:
   target  mae_mean  mae_std  rmse_mean  rmse_std  mape_mean_pct  mape_std_

### Performance in testset

In [ ]:

import os
import numpy as np
import pandas as pd

OUT_DIR = '/content/drive/MyDrive/aim3/CSDI_1e-3_16'

PRED_CSV = os.path.join(
    OUT_DIR, "cv5_test_predictions_pooled.csv"
)

EPS = 1e-8

regions_order = [
    "<1 min",
    "[1,10) min",
    "[10,20) min",
    "[20,30] min",
    "ALL"
]

# ============================================================
# Load predictions
# ============================================================

pred_df = pd.read_csv(PRED_CSV)

if "model" not in pred_df.columns:
    pred_df["model"] = "CSDI_official_architecture"

for col in ["gap_len", "y_true", "y_pred"]:
    pred_df[col] = pd.to_numeric(pred_df[col], errors="coerce")

pred_df = pred_df[
    pred_df["gap_len"].between(1, 360)
    & np.isfinite(pred_df["y_true"])
    & np.isfinite(pred_df["y_pred"])
].copy()

# ============================================================
# Same four gap categories as LSCD
# ============================================================

pred_df["gap_region"] = pd.cut(
    pred_df["gap_len"],
    bins=[0, 11, 119, 239, 360],
    labels=regions_order[:-1],
    include_lowest=True
)

# ============================================================
# Compute MAE and MAPE
# ============================================================

def calculate_metrics(df):
    yt = df["y_true"].to_numpy(dtype=float)
    yp = df["y_pred"].to_numpy(dtype=float)

    abs_err = np.abs(yp - yt)
    valid_mape = np.abs(yt) > EPS

    return pd.Series({
        "MAE": abs_err.mean(),
        "MAPE_pct": (
            100 * np.mean(abs_err[valid_mape] / np.abs(yt[valid_mape]))
            if valid_mape.any() else np.nan
        ),
        "n_points": len(yt),
        "n_mape_points": int(valid_mape.sum())
    })

# ============================================================
# Per-fold metrics
# ============================================================

fold_gap = (
    pred_df
    .groupby(["fold", "target", "model", "gap_region"], observed=True)
    [["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

fold_all = (
    pred_df
    .groupby(["fold", "target", "model"], observed=True)
    [["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

fold_all["gap_region"] = "ALL"

foldwise = pd.concat([fold_gap, fold_all], ignore_index=True)

foldwise["gap_region"] = pd.Categorical(
    foldwise["gap_region"],
    categories=regions_order,
    ordered=True
)

foldwise = foldwise.sort_values(
    ["target", "model", "gap_region", "fold"]
).reset_index(drop=True)

# ============================================================
# Mean ± STD across folds
# ============================================================

summary = (
    foldwise
    .groupby(["target", "model", "gap_region"], observed=True)
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        MAPE_mean_pct=("MAPE_pct", "mean"),
        MAPE_std_pct=("MAPE_pct", "std"),
        n_folds_MAE=("MAE", "count"),
        n_folds_MAPE=("MAPE_pct", "count"),
        total_points=("n_points", "sum")
    )
    .reset_index()
)

# ============================================================
# Manuscript-ready table, same layout as LSCD
# ============================================================

formatted = summary.copy()

for name, mean_col, std_col in [
    ("MAE", "MAE_mean", "MAE_std"),
    ("MAPE(%)", "MAPE_mean_pct", "MAPE_std_pct")
]:
    formatted[name] = formatted.apply(
        lambda r: (
            f"{r[mean_col]:.2f} ± {r[std_col]:.2f}"
            if pd.notna(r[std_col])
            else (
                f"{r[mean_col]:.2f} ± NA"
                if pd.notna(r[mean_col]) else "NA"
            )
        ),
        axis=1
    )

paper = formatted.pivot(
    index=["target", "model"],
    columns="gap_region",
    values=["MAE", "MAPE(%)"]
)

paper = paper.swaplevel(0, 1, axis=1)

paper = paper.reindex(
    columns=pd.MultiIndex.from_product(
        [regions_order, ["MAE", "MAPE(%)"]]
    )
)

paper.columns = [
    f"{gap} | {metric}" for gap, metric in paper.columns
]

paper = paper.reset_index()

# ============================================================
# Pooled metrics
# ============================================================

pool_gap = (
    pred_df
    .groupby(["target", "model", "gap_region"], observed=True)
    [["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

pool_all = (
    pred_df
    .groupby(["target", "model"], observed=True)
    [["y_true", "y_pred"]]
    .apply(calculate_metrics)
    .reset_index()
)

pool_all["gap_region"] = "ALL"

pooled = pd.concat([pool_gap, pool_all], ignore_index=True)

# ============================================================
# Save results
# ============================================================

outputs = {
    "csdi_paper4_foldwise.csv": foldwise,
    "csdi_paper4_mean_std.csv": summary,
    "csdi_paper4_pooled.csv": pooled,
    "csdi_paper4_paper_table.csv": paper,
}

for filename, df in outputs.items():
    path = os.path.join(OUT_DIR, filename)
    df.to_csv(path, index=False)
    print("Saved:", path)

print("\nCSDI FIVE-FOLD MEAN ± STD:")
print(paper.to_string(index=False))


Saved: /content/drive/MyDrive/aim3/CSDI_1e-3_16/csdi_paper4_foldwise.csv
Saved: /content/drive/MyDrive/aim3/CSDI_1e-3_16/csdi_paper4_mean_std.csv
Saved: /content/drive/MyDrive/aim3/CSDI_1e-3_16/csdi_paper4_pooled.csv
Saved: /content/drive/MyDrive/aim3/CSDI_1e-3_16/csdi_paper4_paper_table.csv

CSDI FIVE-FOLD MEAN ± STD:
   target                      model <1 min | MAE <1 min | MAPE(%) [1,10) min | MAE [1,10) min | MAPE(%) [10,20) min | MAE [10,20) min | MAPE(%) [20,30] min | MAE [20,30] min | MAPE(%)   ALL | MAE ALL | MAPE(%)
channel_0 CSDI_official_architecture  0.93 ± 0.17      1.58 ± 0.31      1.30 ± 0.13          2.38 ± 0.52       1.86 ± 0.43           3.32 ± 0.99       2.34 ± 0.52           4.11 ± 1.64 1.54 ± 0.18   2.73 ± 0.50
channel_1 CSDI_official_architecture  1.37 ± 0.22      2.65 ± 1.21      1.68 ± 0.09          2.55 ± 0.21       2.14 ± 0.35           3.32 ± 0.81       5.80 ± 3.14           8.97 ± 4.44 1.87 ± 0.38   3.04 ± 0.89
